# [1단계] 초기 확인용 학습 — 파이프라인 검증 (Colab · A100)

전체 2단계 학습의 **1단계: 파이프라인이 의도대로 도는지 확인하고 baseline을 잡는 pilot**.
- 원본 학습셋(`tsla_train_final.csv`, 보강 전)으로 Fold A 단방향(TSLA학습→NVDA평가).
- 단순 설정: class weight + 일반 CE, LR 2e-5, 5 epoch. (focal·weight 완화 없음)
- 목적: ① 특수토큰/입력 구성/평가 누수 차단이 정상인지 ② 클래스 불균형 문제(C2·C3 희소, C1 과예측)를 **눈으로 확인**.
- 여기서 드러난 문제 → **2단계 `train_colab_2_retrain.ipynb`** 에서 C2/C3 데이터 보강 + focal loss + class weight 완화로 대응.

**실행 순서:** ① 런타임 → 변경 → GPU(A100) ② 위→아래 셀 실행

**업로드 파일 2개** (`data/labeled/`): `tsla_train_final.csv`, `nvda_eval_final.csv`

In [ ]:
!pip install -q -U transformers accelerate scikit-learn

In [ ]:
# 원본 final CSV 2개 업로드 (또는 Google Drive 마운트)
from google.colab import files
up = files.upload()   # tsla_train_final.csv, nvda_eval_final.csv 선택
print(list(up))

In [ ]:
# =============================================================================
#  [1단계] 초기 확인용 학습 — 파이프라인 검증 (Colab·A100 독립 실행)
#  이 파일 + 원본 final CSV 2개만 있으면 됨. 로컬 레포 다른 모듈에 의존하지 않음.
#
#  설계(plan Step4·5):
#   - 입력 = 텍스트 + 주주여부([주주]/[비주주] 토큰 prepend). 주가·뱃지·미래정보 미포함.
#   - KcELECTRA 특수토큰 함정 보정(TemplateProcessing으로 [CLS]/[SEP] 강제).
#   - 불균형: class weight + macro-F1 기준 early stopping(학습셋서 val 분리 → 평가셋 누수 차단).
#   - 벤치마크: Fold A 단방향(TSLA학습→NVDA평가)에서 3모델 비교 → macro-F1 최고 선택.
#   - 베이스라인: 다수클래스 / TF-IDF+LogReg.
#   - 저장: 결과표 runs/results_A_1단계.md + 최고 모델 runs/A_{best}/model (+zip).
#  ※ 이 단계는 '확인용' — 단순 설정으로 baseline만 본다. 튜닝은 2단계에서.
# =============================================================================
import os
import shutil
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, Trainer, TrainingArguments,
                          EarlyStoppingCallback)

# ── CONFIG ───────────────────────────────────────────────────────────────────
DATA_DIR = "."                  # 업로드한 CSV 위치
FOLDS = {
    "A": dict(train="tsla_train_final.csv", test="nvda_eval_final.csv",
              note="TSLA학습(원본)→NVDA평가"),
}
MODELS = {
    "KcELECTRA":    "beomi/KcELECTRA-base",        # 주력(댓글 도메인)
    "KLUE-RoBERTa": "klue/roberta-base",           # 한국어 NLU 표준 레퍼런스
    "KR-FinBERT":   "snunlp/KR-FinBert",           # 금융 도메인(ablation 성격)
}
NUM_LABELS = 4
LABELS, NAMES = [0, 1, 2, 3], ["C0예측없음", "C1실패", "C2방향적중", "C3날짜적중"]
MAX_LEN = 512          # plan 확정. 짧은 댓글은 dynamic padding으로 효율 처리
VAL_SIZE = 0.15        # 학습셋에서 떼는 검증 비율(early stopping 용, 평가셋과 분리)
EPOCHS = 5             # 확인용 — 짧게
BATCH = 32
LR = 2e-5              # 기본값(튜닝 전)
SEED = 42
HOLDER_TOKENS = ["[주주]", "[비주주]"]
OUT_DIR = "runs"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# ── 데이터 ───────────────────────────────────────────────────────────────────
def make_inputs(df: pd.DataFrame) -> list:
    """텍스트 앞에 [주주]/[비주주] 토큰 prepend (입력 = 텍스트 + 주주여부)."""
    holder = df["주주_여부"].astype(str).str.lower().eq("true").map(
        {True: "[주주]", False: "[비주주]"})
    return (holder + " " + df["text"].astype(str)).tolist()


def load_fold(fold: str):
    f = FOLDS[fold]
    tr = pd.read_csv(os.path.join(DATA_DIR, f["train"]), encoding="utf-8-sig")
    te = pd.read_csv(os.path.join(DATA_DIR, f["test"]), encoding="utf-8-sig")
    for d in (tr, te):
        d.dropna(subset=["Class", "text"], inplace=True)
        d["Class"] = d["Class"].astype(int)
    return tr, te, f["note"]


class TextDS(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tok):
        self.enc = tok(texts, truncation=True, max_length=MAX_LEN)
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        item = {k: self.enc[k][i] for k in self.enc}
        item["labels"] = int(self.labels[i])
        return item


# ── 토크나이저(특수토큰 함정 보정) ───────────────────────────────────────────
def build_tokenizer(model_name: str):
    tok = AutoTokenizer.from_pretrained(model_name)
    tok.add_special_tokens({"additional_special_tokens": HOLDER_TOKENS})
    # KcELECTRA 등 post_processor 없는 토크나이저: [CLS]/[SEP] 자동추가 안 됨 → 강제
    ids = tok("테스트")["input_ids"]
    if tok.cls_token_id is not None and (len(ids) == 0 or ids[0] != tok.cls_token_id):
        from tokenizers.processors import TemplateProcessing
        tok._tokenizer.post_processor = TemplateProcessing(
            single=f"{tok.cls_token} $A {tok.sep_token}",
            pair=f"{tok.cls_token} $A {tok.sep_token} $B:1 {tok.sep_token}:1",
            special_tokens=[(tok.cls_token, tok.cls_token_id),
                            (tok.sep_token, tok.sep_token_id)])
        chk = tok("테스트")["input_ids"]
        assert chk[0] == tok.cls_token_id and chk[-1] == tok.sep_token_id, "CLS/SEP 보정 실패"
        print(f"    [특수토큰 보정] {model_name}: [CLS]/[SEP] TemplateProcessing 적용")
    return tok


class WeightedTrainer(Trainer):
    def __init__(self, *a, class_weights=None, **k):
        super().__init__(*a, **k)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels")
        out = model(**inputs)
        loss = F.cross_entropy(out.logits, labels,
                               weight=self.class_weights.to(out.logits.device))
        return (loss, out) if return_outputs else loss


def metrics_fn(p):
    preds = p.predictions.argmax(-1)
    return {"macro_f1": f1_score(p.label_ids, preds, average="macro"),
            "acc": accuracy_score(p.label_ids, preds)}


# 결과표 누적 버퍼 — report() 호출마다 한 행씩 쌓고, 끝에 .md로 저장
RESULTS_ROWS = []


def report(name, y_true, y_pred):
    rep = classification_report(y_true, y_pred, labels=LABELS, target_names=NAMES,
                                digits=3, zero_division=0, output_dict=True)
    mf1, acc = rep["macro avg"]["f1-score"], rep["accuracy"]
    print(f"\n  ── {name} ──  macro-F1 = {mf1:.4f}  acc = {acc:.4f}")
    print(classification_report(y_true, y_pred, labels=LABELS, target_names=NAMES,
                                digits=3, zero_division=0))
    print("  혼동행렬 (행=실제, 열=예측):\n", confusion_matrix(y_true, y_pred, labels=LABELS))
    RESULTS_ROWS.append({"name": name, "macro_f1": mf1, "acc": acc,
                         "per_f1": [rep[n]["f1-score"] for n in NAMES]})
    return mf1


def save_results_md(path, title, support=None):
    """RESULTS_ROWS를 macro-F1 내림차순 마크다운 표로 저장."""
    cols = ["모델 / 베이스라인", "macro-F1", "acc"] + [f"{n}-F1" for n in NAMES]
    lines = [f"# {title}", ""]
    if support is not None:
        lines += [f"- 평가셋 클래스 support = {support}", ""]
    lines += ["| " + " | ".join(cols) + " |",
              "|" + "|".join(["---"] * len(cols)) + "|"]
    for r in sorted(RESULTS_ROWS, key=lambda d: -d["macro_f1"]):
        cells = [r["name"], f'{r["macro_f1"]:.3f}', f'{r["acc"]:.3f}'] + \
                [f'{v:.3f}' for v in r["per_f1"]]
        lines.append("| " + " | ".join(cells) + " |")
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n")
    print("결과표 저장:", path)
    return path


# ── 단일 모델 fine-tuning + 교차평가 ─────────────────────────────────────────
def run_model(model_name, hf_id, tr_df, te_df, note, save_model=True):
    tok = build_tokenizer(hf_id)
    Xtr = make_inputs(tr_df); ytr = tr_df["Class"].to_numpy()
    Xte = make_inputs(te_df); yte = te_df["Class"].to_numpy()
    # 학습셋 내부 train/val(early stopping용) — 평가셋(타 종목)은 손대지 않음
    Xt, Xv, yt, yv = train_test_split(Xtr, ytr, test_size=VAL_SIZE,
                                      stratify=ytr, random_state=SEED)
    ds_tr, ds_va, ds_te = TextDS(Xt, yt, tok), TextDS(Xv, yv, tok), TextDS(Xte, yte, tok)

    model = AutoModelForSequenceClassification.from_pretrained(hf_id, num_labels=NUM_LABELS)
    model.resize_token_embeddings(len(tok))     # [주주]/[비주주] 추가분 반영

    cw = compute_class_weight("balanced", classes=np.arange(NUM_LABELS), y=yt)
    cw = torch.tensor(cw, dtype=torch.float)

    args = TrainingArguments(
        output_dir=os.path.join(OUT_DIR, f"A_{model_name}"),
        num_train_epochs=EPOCHS, learning_rate=LR,
        per_device_train_batch_size=BATCH, per_device_eval_batch_size=64,
        eval_strategy="epoch", save_strategy="epoch", logging_steps=50,
        load_best_model_at_end=True, metric_for_best_model="macro_f1",
        greater_is_better=True, save_total_limit=1, seed=SEED, report_to="none",
        fp16=torch.cuda.is_available())
    trainer = WeightedTrainer(
        model=model, args=args, train_dataset=ds_tr, eval_dataset=ds_va,
        data_collator=DataCollatorWithPadding(tok), compute_metrics=metrics_fn,
        class_weights=cw, callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
    trainer.train()
    pred = trainer.predict(ds_te).predictions.argmax(-1)
    if save_model:                              # best epoch 모델 + 토크나이저 저장
        save_dir = os.path.join(OUT_DIR, f"A_{model_name}", "model")
        trainer.save_model(save_dir); tok.save_pretrained(save_dir)
        print(f"    모델 저장: {save_dir}")
    return report(f"[A] {model_name} ({note})", yte, pred)


# ── 베이스라인 ───────────────────────────────────────────────────────────────
def baselines(tr_df, te_df):
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    Xtr, ytr = make_inputs(tr_df), tr_df["Class"].to_numpy()
    Xte, yte = make_inputs(te_df), te_df["Class"].to_numpy()
    # 다수 클래스
    maj = np.bincount(ytr, minlength=NUM_LABELS).argmax()
    report(f"[A] 베이스라인-다수클래스(={maj})", yte, np.full_like(yte, maj))
    # TF-IDF + LogReg
    vec = TfidfVectorizer(max_features=30000, ngram_range=(1, 2), min_df=2)
    Ztr, Zte = vec.fit_transform(Xtr), vec.transform(Xte)
    lr = LogisticRegression(max_iter=1000, class_weight="balanced")
    lr.fit(Ztr, ytr)
    report("[A] 베이스라인-TFIDF+LogReg", yte, lr.predict(Zte))


# ── 메인 ─────────────────────────────────────────────────────────────────────
def main():
    print(f"device={DEVICE}")
    os.makedirs(OUT_DIR, exist_ok=True)
    trA, teA, noteA = load_fold("A")
    print("학습셋 Class 분포(원본 A-train):", dict(trA["Class"].value_counts().sort_index()))
    print("평가셋 Class 분포(NVDA):", dict(teA["Class"].value_counts().sort_index()))

    # [1] 벤치마크: Fold A 단방향(TSLA학습→NVDA평가)에서 3모델 비교 → 최고 선택
    print("\n===== [1] 모델 벤치마크 (Fold A: TSLA학습→NVDA평가) =====")
    scores = {}
    for name, hf_id in MODELS.items():
        try:
            scores[name] = run_model(name, hf_id, trA, teA, noteA)
        except Exception as e:
            print(f"  ⚠️ {name} 실패: {e}")
    best = max(scores, key=scores.get)
    print("\n벤치마크 macro-F1:", {k: round(v, 4) for k, v in scores.items()})
    print(f"→ 최고 모델: {best} (macro-F1 {scores[best]:.4f})")

    # [2] 베이스라인 (Fold A)
    print("\n===== [2] 베이스라인 (Fold A) =====")
    baselines(trA, teA)

    # [3] 저장 — 결과표(.md) + 최고 모델 zip
    print("\n===== [3] 저장 =====")
    support = dict(zip(NAMES, np.bincount(teA["Class"], minlength=NUM_LABELS)))
    md_path = save_results_md(os.path.join(OUT_DIR, "results_A_1단계.md"),
                              "Fold A 결과 — TSLA학습(원본)→NVDA평가 · 1단계(검증)", support)
    print(open(md_path, encoding="utf-8").read())
    best_dir = os.path.join(OUT_DIR, f"A_{best}", "model")
    if os.path.isdir(best_dir):
        zip_path = shutil.make_archive(os.path.join(OUT_DIR, f"best_{best}"), "zip", best_dir)
        print(f"최고 모델({best}) zip: {zip_path}  ({os.path.getsize(zip_path)/1e6:.0f} MB)")
    # 다운로드 원하면: from google.colab import files; files.download(md_path); files.download(zip_path)

    print("\n[1단계 완료] baseline·클래스별 PR·혼동행렬 확인 → C2/C3 희소·C1 과예측 여부 점검.")
    print("다음: 2단계(train_colab_2_retrain.ipynb)에서 데이터 보강 + focal/weight 완화로 재학습.")


if __name__ == "__main__":
    main()